In [1]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    Provider("OpenAI", "OPENAI_API_KEY", True, None, "gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )


def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [2]:
def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content

In [4]:
prompt = "Who was the PM of UK before 2020 ? Answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")




Using Groq provider
The Prime Minister of the United Kingdom before 2020 was Boris Johnson, who assumed office on 24 July 2019.


## Modify the implementation of providers to add a OPEN Router provider as well. 